# 01 - Time Series Preparation

> Convert transactional data to monthly time series, analyze trend/seasonality, engineer features, and create chronological train/test split.

**Pipeline**: sales_clean.csv -> Monthly Aggregation -> Stationarity Tests -> Feature Engineering -> Train/Test Split

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
COLORS = {'primary':'#003B73','secondary':'#0074B7','success':'#27AE60','alert':'#BF212F'}
print("Libraries loaded")

: 

## 1. Load Raw Transactional Data

In [ ]:
df = pd.read_csv('../../02_Dataset/cleaned_data/sales_clean.csv', parse_dates=['order_date'])
print(f"Dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Date Range: {df['order_date'].min().date()} to {df['order_date'].max().date()}")
print(f"Unique Orders: {df['order_id'].nunique():,}")

## 2. Aggregate to Monthly Sales

In [ ]:
monthly = df.resample('ME', on='order_date')['sales'].sum().reset_index()
monthly.rename(columns={'order_date':'ds','sales':'y'}, inplace=True)
monthly['ds'] = monthly['ds'].dt.to_period('M').dt.to_timestamp()
print(f"Monthly series: {len(monthly)} months")
print(f"Date Range: {monthly['ds'].min().date()} to {monthly['ds'].max().date()}")
monthly.head()

## 3. Check for Missing Months

In [ ]:
full_range = pd.date_range(start=monthly['ds'].min(), end=monthly['ds'].max(), freq='MS')
missing = set(full_range) - set(monthly['ds'])
if missing:
    print(f"WARNING: {len(missing)} missing months: {sorted(missing)}")
else:
    print("No missing months - continuous monthly series confirmed")
print(f"Total months: {len(monthly)}")

## 4. Time Series Visualization

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

axes[0].fill_between(monthly['ds'], monthly['y'], alpha=0.3, color=COLORS['primary'])
axes[0].plot(monthly['ds'], monthly['y'], color=COLORS['primary'], linewidth=2, marker='o', markersize=4)
axes[0].set_title('Monthly Sales Time Series (2011-2014)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Sales ($)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))

rolling_mean = monthly['y'].rolling(window=3).mean()
rolling_std = monthly['y'].rolling(window=3).std()
axes[1].plot(monthly['ds'], monthly['y'], label='Original', color=COLORS['primary'], alpha=0.5)
axes[1].plot(monthly['ds'], rolling_mean, label='3-Month Rolling Mean', color=COLORS['alert'], linewidth=2)
axes[1].fill_between(monthly['ds'], rolling_mean - rolling_std, rolling_mean + rolling_std, alpha=0.15, color=COLORS['alert'])
axes[1].set_title('Rolling Mean & Standard Deviation', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Sales ($)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../visualizations/sales_time_series.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Seasonal Decomposition

In [ ]:
ts = monthly.set_index('ds')['y']
decomp = seasonal_decompose(ts, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(16, 14))
decomp.observed.plot(ax=axes[0], color=COLORS['primary']); axes[0].set_title('Observed', fontweight='bold')
decomp.trend.plot(ax=axes[1], color=COLORS['success']); axes[1].set_title('Trend', fontweight='bold')
decomp.seasonal.plot(ax=axes[2], color=COLORS['alert']); axes[2].set_title('Seasonality', fontweight='bold')
decomp.resid.plot(ax=axes[3], color=COLORS['secondary']); axes[3].set_title('Residuals', fontweight='bold')

plt.suptitle('Seasonal Decomposition (Additive, Period=12)', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../visualizations/seasonal_decomposition.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Stationarity Test (ADF)

In [ ]:
def adf_test(series, title=''):
    result = adfuller(series, autolag='AIC')
    print(f'ADF Test: {title}')
    print(f'  ADF Statistic: {result[0]:.4f}')
    print(f'  p-value:       {result[1]:.4f}')
    print(f'  Critical Values:')
    for k, v in result[4].items():
        print(f'    {k}: {v:.4f}')
    print(f'  Stationary: {"YES" if result[1] < 0.05 else "NO (needs differencing)"}')
    print()

adf_test(monthly['y'], 'Original Series')
adf_test(monthly['y'].diff().dropna(), 'First Difference')

## 7. Feature Engineering for ML Models

In [ ]:
ml_data = monthly.copy()
ml_data['month'] = ml_data['ds'].dt.month
ml_data['quarter'] = ml_data['ds'].dt.quarter
ml_data['year'] = ml_data['ds'].dt.year
ml_data['trend'] = np.arange(len(ml_data))

# Lag features
ml_data['lag_1'] = ml_data['y'].shift(1)
ml_data['lag_2'] = ml_data['y'].shift(2)
ml_data['lag_3'] = ml_data['y'].shift(3)
ml_data['lag_12'] = ml_data['y'].shift(12)

# Rolling statistics
ml_data['rolling_mean_3'] = ml_data['y'].shift(1).rolling(3).mean()
ml_data['rolling_mean_6'] = ml_data['y'].shift(1).rolling(6).mean()
ml_data['rolling_std_3'] = ml_data['y'].shift(1).rolling(3).std()

# Drop NaN rows caused by shift/rolling
ml_data_clean = ml_data.dropna().reset_index(drop=True)
print(f"ML dataset: {ml_data_clean.shape[0]} rows x {ml_data_clean.shape[1]} columns")
print(f"Features: {[c for c in ml_data_clean.columns if c not in ['ds','y']]}")
ml_data_clean.head()

## 8. Chronological Train/Test Split

> **IMPORTANT**: We use a chronological split (NOT random) to prevent time-series data leakage.
> - **Train**: Jan 2012 - Dec 2013 (after lag features drop early months)
> - **Test**: Jan 2014 - Dec 2014 (12 months)

In [ ]:
SPLIT_DATE = '2014-01-01'

# For Prophet/ARIMA (just ds and y)
train_ts = monthly[monthly['ds'] < SPLIT_DATE].copy()
test_ts = monthly[monthly['ds'] >= SPLIT_DATE].copy()

# For ML models (with engineered features)
train_ml = ml_data_clean[ml_data_clean['ds'] < SPLIT_DATE].copy()
test_ml = ml_data_clean[ml_data_clean['ds'] >= SPLIT_DATE].copy()

FEATURES = ['month','quarter','year','trend','lag_1','lag_2','lag_3','lag_12',
            'rolling_mean_3','rolling_mean_6','rolling_std_3']

print(f"Time Series Split:")
print(f"  Train: {train_ts['ds'].min().date()} to {train_ts['ds'].max().date()} ({len(train_ts)} months)")
print(f"  Test:  {test_ts['ds'].min().date()} to {test_ts['ds'].max().date()} ({len(test_ts)} months)")
print(f"\nML Split (after feature engineering):")
print(f"  Train: {train_ml['ds'].min().date()} to {train_ml['ds'].max().date()} ({len(train_ml)} months)")
print(f"  Test:  {test_ml['ds'].min().date()} to {test_ml['ds'].max().date()} ({len(test_ml)} months)")

## 9. Save Prepared Data

In [ ]:
# Save all prepared datasets for downstream notebooks
monthly.to_csv('../data/monthly_sales.csv', index=False)
train_ts.to_csv('../data/train_ts.csv', index=False)
test_ts.to_csv('../data/test_ts.csv', index=False)
ml_data_clean.to_csv('../data/ml_features.csv', index=False)
train_ml.to_csv('../data/train_ml.csv', index=False)
test_ml.to_csv('../data/test_ml.csv', index=False)

print("Saved to ../data/:")
print("  monthly_sales.csv  - Full monthly time series (ds, y)")
print("  train_ts.csv       - Train set for ARIMA/Prophet")
print("  test_ts.csv        - Test set for ARIMA/Prophet")
print("  ml_features.csv    - Full ML feature dataset")
print("  train_ml.csv       - Train set for XGBoost/RF")
print("  test_ml.csv        - Test set for XGBoost/RF")
print("\nProceed to 02_ARIMA_Model.ipynb")